# 第 10 章 ニューラルネットワーク

直線では分けられない XOR を、隠れ層を積むことで解きます。学習は誤差逆伝播法で行います。

対応する記事: [第 10 章 ニューラルネットワーク（Jupyter Notebook（Python） の言語版）](../../../docs/article/grokking-machine-learning/python/ch10.md)

実装本体: `apps/grokking-ml-python/src/`

## セットアップ

実装本体（`../src/grokking_ml/`）を読み込みます。**ノートブックにコードを複製せず、記事と同じ実装をそのまま使います。**

```bash
cd apps/grokking-ml-python
uv sync
uv run jupyter lab notebooks/
```

In [1]:
import pathlib
import sys

sys.path.insert(0, str(pathlib.Path.cwd().parent / "src"))

from grokking_ml.ch10_neural_networks import *

## XOR は直線で分けられない

対角線上の 2 点が同じクラスなので、**1 本の直線では絶対に分けられません。**

In [2]:
points = [(0.0, 0.0), (0.0, 1.0), (1.0, 0.0), (1.0, 1.0)]
labels = [0, 1, 1, 0]

for point, label in zip(points, labels):
    print(f"({point[0]:.0f}, {point[1]:.0f}) → {label}")

(0, 0) → 0
(0, 1) → 1
(1, 0) → 1
(1, 1) → 0


## 隠れ層の幅を変えて比べる

**隠れ層があっても、ニューロンが 1 つでは足りません。** 実質「直線を 1 本引いてから変換する」だけなので、表現力はロジスティック回帰と変わらないからです。

**「層を足せば強くなる」ではなく「十分な幅の隠れ層が要る」** ということです。

In [3]:
for hidden in [1, 2, 4]:
    m, losses = train(points, labels, hidden_size=hidden, epochs=20000, seed=0)
    print(f"隠れ層 {hidden} ニューロン  正解率 {accuracy(m, points, labels):.2f}  "
          f"損失 {losses[0]:.4f} → {losses[-1]:.4f}")

隠れ層 1 ニューロン  正解率 0.75  損失 0.7035 → 0.5192


隠れ層 2 ニューロン  正解率 1.00  損失 0.6956 → 0.0010


隠れ層 4 ニューロン  正解率 1.00  損失 0.7518 → 0.0009


## 学習後の予測

隠れ層 4 ニューロンなら、**4 点すべてを 0.999 以上の確信で当てられます。**

In [4]:
model, losses = train(points, labels, hidden_size=4, epochs=20000, seed=0)

for point, label in zip(points, labels):
    probability = model.predict_probability(point)
    print(f"({point[0]:.0f}, {point[1]:.0f}) 正解={label}  予測確率 {probability:.4f}")

(0, 0) 正解=0  予測確率 0.0007
(0, 1) 正解=1  予測確率 0.9992
(1, 0) 正解=1  予測確率 0.9992
(1, 1) 正解=0  予測確率 0.0015


## 勾配消失

シグモイドの微分は最大 0.25、両端では 0 に近づきます。**層を深く積むとこの小さな値が掛け合わされ、入力側の層がほとんど学習しなくなります。**

現代のネットワークが ReLU を使う理由がここにあります。

In [5]:
print(f"{'出力':>8} {'微分':>10}")
for output in [0.001, 0.1, 0.5, 0.9, 0.999]:
    print(f"{output:>8.3f} {sigmoid_derivative(output):>10.6f}")

print()
print(f"10 層積んだときの積 {0.25 ** 10:.2e}")

      出力         微分
   0.001   0.000999
   0.100   0.090000
   0.500   0.250000
   0.900   0.090000
   0.999   0.000999

10 層積んだときの積 9.54e-07


## 試してみる: 学習の途中経過

損失がどう下がるかを見ます。**XOR は最初しばらく停滞してから、あるところで急に解けます。**

In [6]:
for epoch in range(0, 20000, 2000):
    bar = "#" * int(losses[epoch] * 50)
    print(f"epoch {epoch:6d}  損失 {losses[epoch]:.4f}  {bar}")

epoch      0  損失 0.7518  #####################################
epoch   2000  損失 0.0345  #
epoch   4000  損失 0.0075  
epoch   6000  損失 0.0041  
epoch   8000  損失 0.0028  
epoch  10000  損失 0.0021  
epoch  12000  損失 0.0017  
epoch  14000  損失 0.0014  
epoch  16000  損失 0.0012  
epoch  18000  損失 0.0011  
